# 03 · SFT

Учим на эталонных метках: $\mathcal{L} = -\frac{1}{T}\sum_t \log \pi_\theta(y_t \mid x, y_{<t})$
только по токенам ответа, промпт под $-100$. Ответ здесь короткий, поэтому SFT учит в первую очередь
решение, а не манеру. Адаптер LoRA тот же, что у ассистента.

In [ ]:
import sys
sys.path.insert(0, "../..")

from src import data, infer
from src import filter as F

from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

model, tokenizer = infer.load_model()
train = data.to_sft(F.load("train")).select_columns(["prompt", "completion"])

In [ ]:
lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    # attention and MLP projections of the language stack; the vision tower is excluded, there are no images
    target_modules=r"^(?!.*(visual|vision)).*(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$",
    use_rslora=True,
    task_type="CAUSAL_LM",
)

config = SFTConfig(
    output_dir=str(F.RUNS / "sft"),
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_steps=0.05,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_length=1024,
    completion_only_loss=True,
    logging_steps=5,
    save_strategy="no",
    report_to=[],
    seed=42,
)
trainer = SFTTrainer(model=model, args=config, train_dataset=train, processing_class=tokenizer, peft_config=lora)
trainer.model.print_trainable_parameters()

example = trainer.train_dataset[0]
trained = [l != -100 for l in example["labels"]]
print("под градиентом:", repr(tokenizer.decode([i for i, t in zip(example["input_ids"], trained) if t])))

In [ ]:
history = trainer.train()
tuned = trainer.model
tuned.save_pretrained(F.RUNS / "sft-adapter")
print(f"loss {history.training_loss:.3f} | {infer.free(trainer)}")
del trainer

F.evaluate(tuned, tokenizer, "sft", note="LoRA SFT, 3 epochs, lr 1e-4")
F.show()

In [ ]:
print("что ещё ошибается после SFT")
for e in F.errors("sft")[:10]:
    print(f"  {e['truth']:5} {e['category'] or '':22} {e['request'][:60]:60} → {e['answer'].splitlines()[0]}")